# 99. Конфигурация Excel Loader

Этот ноутбук помогает:
1. **Сгенерировать Excel шаблон** с цветовой кодировкой и YAML конфигом
2. **Обновить Excel шаблон** (добавить все метрики из БД)
3. **Создать упрощенный шаблон** для Standard/Lite моделей (только канонические метрики)
4. **Экспортировать данные из БД в Excel шаблон** (для ручной корректировки)
5. **Загрузить данные из Excel в БД** с автоматическим обнаружением кастомных метрик
6. **Обновить dictionary_metrics** автоматически при обнаружении новых метрик
7. Проверить структуру Excel-файла и увидеть алиасы метрик
8. Сгенерировать YAML-фрагменты для `configs/excel_loader.yaml`

**Доступные шаблоны:**
- `template_UNIFIED_All_Data.xlsx` - полный шаблон для всех моделей (Standard, Custom, Lite) - содержит все метрики из БД
- `template_STANDARD_LITE.xlsx` - упрощенный шаблон только для Standard и Lite моделей - содержит только канонические метрики

**Рабочий процесс:**
1. Генерация шаблона с YAML конфигом (цветовая кодировка обязательных метрик)
2. Выбор шаблона (полный или упрощенный)
3. Экспорт данных из БД → Excel (если данные уже в БД)
4. Ручная корректировка данных в Excel (можно добавить кастомные метрики)
5. Загрузка данных из Excel → БД с автоматическим обнаружением кастомных метрик
6. Автоматическое обновление dictionary_metrics (опционально)
7. Проверка и валидация загруженных данных

**⭐ Новые возможности:**
- **Цветовая кодировка метрик**: 🟡 Желтый = обязательные, 🔵 Голубой = расчетные, ⚪ Белый = опциональные
- **Автоматическое обнаружение кастомных метрик**: при загрузке Excel автоматически обнаруживаются метрики, которых нет в dictionary_metrics
- **Автоматическое предложение алиасов**: система предлагает похожие метрики для новых кастомных метрик
- **Автоматическое обновление dictionary_metrics**: можно автоматически добавить новые метрики в словарь

**⭐ Новые breakdown метрики (опциональные)**:
Excel Loader теперь поддерживает детализацию метрик для более точного моделирования:
- **IS**: `revenue_product`, `revenue_service`, `cogs_materials`, `cogs_labor`, `sga_selling`, `current_tax_expense`, `deferred_tax_expense`, `gross_margin`, `operating_margin`
- **BS**: `accounts_receivable_gross`, `allowance_for_doubtful_accounts`, `inventory_raw_materials`, `inventory_wip`, `inventory_finished_goods`, `rou_finance_asset`, `rou_operating_asset`, `current_finance_lease_liability`, `current_operating_lease_liability`
- **CF**: `change_ar`, `change_inventory`, `change_ap`, `lease_payments_cfo_operating`, `finance_lease_principal`, `debt_issuance`, `debt_repayment`

Все новые метрики опциональны и могут быть добавлены в Excel шаблон для более детального моделирования.



In [ ]:
from pathlib import Path
from collections import Counter
import sys
import subprocess

import pandas as pd
import yaml

# Автоматическое определение компании из пути ноутбука
current_dir = Path.cwd()
if 'companies' in current_dir.parts:
    companies_idx = current_dir.parts.index('companies')
    if companies_idx + 1 < len(current_dir.parts):
        COMPANY = current_dir.parts[companies_idx + 1]
    else:
        COMPANY = 'us_steel'  # fallback
else:
    COMPANY = 'us_steel'  # fallback

# Определение корня проекта
if (current_dir / 'engine').exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / 'engine').exists():
    PROJECT_ROOT = current_dir.parent
elif (current_dir.parent.parent / 'engine').exists():
    PROJECT_ROOT = current_dir.parent.parent
else:
    PROJECT_ROOT = current_dir.parent.parent.parent

sys.path.insert(0, str(PROJECT_ROOT))

# Настройки путей
COMPANY_ROOT = PROJECT_ROOT / 'companies' / COMPANY
EXCEL_INPUT = COMPANY_ROOT / "data" / "excel" / f"{COMPANY}_input.xlsx"
EXCEL_FILLED = COMPANY_ROOT / "data" / "excel" / f"{COMPANY}_filled.xlsx"
CONFIG_PATH = COMPANY_ROOT / "configs" / "excel_loader.yaml"
TEMPLATE_PATH = PROJECT_ROOT / "templates" / "excel_templates" / "template_UNIFIED_All_Data.xlsx"

print(f"📁 Компания: {COMPANY}")
print(f"📁 Корень проекта: {PROJECT_ROOT}")
print(f"📁 Excel input: {EXCEL_INPUT}")
print(f"📁 Excel filled: {EXCEL_FILLED}")
print(f"📁 Config: {CONFIG_PATH}")
print(f"📁 Template: {TEMPLATE_PATH}")


In [ ]:
## 0️⃣ Генерация Excel шаблона с цветовой кодировкой

Генерирует новый Excel шаблон с цветовой кодировкой обязательных метрик и загрузкой combine_from правил из YAML конфига.

# Генерация Excel шаблона с цветовой кодировкой и YAML конфигом
TEMPLATE_DIR = PROJECT_ROOT / "templates" / "excel_templates"
TEMPLATE_DIR.mkdir(parents=True, exist_ok=True)

generate_cmd = [
    "python3",
    str(PROJECT_ROOT / "tools" / "excel_template_generator.py"),
    "--output_dir", str(TEMPLATE_DIR),
    "--yaml_config", str(CONFIG_PATH) if CONFIG_PATH.exists() else ""
]

# Убираем пустой yaml_config, если конфиг не найден
if not CONFIG_PATH.exists():
    generate_cmd = generate_cmd[:-2]

print("🎨 Генерация Excel шаблона с цветовой кодировкой...")
print(f"Команда: {' '.join(generate_cmd)}")
print(f"📄 Выходной шаблон: {TEMPLATE_PATH}")
print(f"📄 YAML конфиг: {CONFIG_PATH if CONFIG_PATH.exists() else 'не используется'}")
print("\n⚠️ Для реальной генерации раскомментируйте код ниже:")

# Раскомментируйте для реальной генерации:
# env = os.environ.copy()
# env['PYTHONPATH'] = str(PROJECT_ROOT) + ':' + env.get('PYTHONPATH', '')
# result = subprocess.run(generate_cmd, cwd=PROJECT_ROOT, env=env, capture_output=True, text=True)
# print(result.stdout)
# if result.returncode != 0:
#     print("❌ Ошибка при генерации:")
#     print(result.stderr)
# else:
#     print("✅ Шаблон успешно сгенерирован!")
#     print("   🟡 Желтая подсветка = обязательные метрики")
#     print("   🔵 Голубая подсветка = расчетные метрики (combine_from)")
#     print("   ⚪ Белая = опциональные метрики")


## 0️⃣.1 Автоматическая генерация excel_loader.yaml

Генерация полного  на основе канонических метрик и схемы БД.

In [ ]:
# Автоматическая генерация excel_loader.yaml
import subprocess
import os

generate_yaml_cmd = [
    "python3",
    str(PROJECT_ROOT / "tools" / "generate_excel_loader_yaml.py"),
    "--company", COMPANY,
    "--output", str(CONFIG_PATH)
]

print("🔄 Генерация excel_loader.yaml...")
print(f"Команда: {' '.join(generate_yaml_cmd)}")
print(f"📄 Выходной файл: {CONFIG_PATH}")
print("\n⚠️ Для реальной генерации раскомментируйте код ниже:")

# Раскомментируйте для реальной генерации:
# env = os.environ.copy()
# env['PYTHONPATH'] = str(PROJECT_ROOT) + ':' + env.get('PYTHONPATH', '')
# result = subprocess.run(generate_yaml_cmd, cwd=PROJECT_ROOT, env=env, capture_output=True, text=True)
# print(result.stdout)
# if result.returncode != 0:
#     print("❌ Ошибка при генерации:")
#     print(result.stderr)
# else:
#     print("✅ excel_loader.yaml успешно сгенерирован!")
#     print(f"   Проверьте файл: {CONFIG_PATH}")

## 1️⃣ Экспорт данных из БД в Excel шаблон

Если данные уже загружены в БД (например, из EDGAR), можно экспортировать их в Excel для ручной корректировки.


## 0.1 Обновление Excel шаблона: добавление всех метрик из БД

ВАЖНО: Это обновит сам шаблон, добавив все метрики из БД (если используется старый скрипт обновления).


## 0.2 Создание упрощенного шаблона для Standard/Lite моделей

Создает упрощенный шаблон, содержащий только канонические метрики, необходимые для Standard и Lite моделей.


In [ ]:
# Создание упрощенного шаблона для Standard/Lite моделей
# Этот шаблон содержит только канонические метрики

STANDARD_TEMPLATE_PATH = PROJECT_ROOT / "templates" / "excel_templates" / "template_STANDARD_LITE.xlsx"

if TEMPLATE_PATH.exists():
    create_standard_cmd = [
        "python3",
        str(PROJECT_ROOT / "tools" / "create_standard_template.py"),
        "--template", str(TEMPLATE_PATH),
        "--output", str(STANDARD_TEMPLATE_PATH)
    ]
    
    print("🔧 Создание упрощенного шаблона для Standard/Lite...")
    print(f"Команда: {' '.join(create_standard_cmd)}")
    print(f"📄 Исходный шаблон: {TEMPLATE_PATH}")
    print(f"📄 Выходной шаблон: {STANDARD_TEMPLATE_PATH}")
    print("\n⚠️ Для реального создания раскомментируйте код ниже:")
    
    # Раскомментируйте для реального создания:
    # env = os.environ.copy()
    # env['PYTHONPATH'] = str(PROJECT_ROOT) + ':' + env.get('PYTHONPATH', '')
    # result = subprocess.run(create_standard_cmd, cwd=PROJECT_ROOT, env=env, capture_output=True, text=True)
    # print(result.stdout)
    # if result.returncode != 0:
    #     print("❌ Ошибка при создании:")
    #     print(result.stderr)
    # else:
    #     print("✅ Упрощенный шаблон успешно создан!")
    #     print(f"   Используйте его для Standard и Lite моделей")
else:
    print(f"⚠️ Исходный шаблон не найден: {TEMPLATE_PATH}")


In [ ]:
# Экспорт данных из БД в Excel шаблон
import os

if TEMPLATE_PATH.exists():
    export_cmd = [
        "python3", 
        str(PROJECT_ROOT / "tools" / "export_data_to_excel_template.py"),
        "--company", COMPANY,
        "--output", str(EXCEL_FILLED),
        "--template", str(TEMPLATE_PATH)
    ]
    
    print("🚀 Запуск экспорта данных из БД в Excel...")
    print(f"Команда: {' '.join(export_cmd)}")
    
    env = os.environ.copy()
    env['PYTHONPATH'] = str(PROJECT_ROOT) + ':' + env.get('PYTHONPATH', '')
    
    result = subprocess.run(export_cmd, cwd=PROJECT_ROOT, env=env, capture_output=True, text=True)
    
    if result.returncode == 0:
        print("✅ Экспорт выполнен успешно!")
        if EXCEL_FILLED.exists():
            print(f"📄 Файл создан: {EXCEL_FILLED}")
            print(f"   Размер: {EXCEL_FILLED.stat().st_size / 1024:.1f} KB")
    else:
        print("❌ Ошибка при экспорте:")
        print(result.stderr)
else:
    print(f"⚠️ Шаблон не найден: {TEMPLATE_PATH}")


## 2️⃣ Загрузка данных из Excel в БД

После ручной корректировки данных в Excel, загрузите их обратно в БД.


In [ ]:
# Загрузка данных из Excel в БД с автоматическим обнаружением кастомных метрик
# ВАЖНО: Используйте EXCEL_FILLED или EXCEL_INPUT в зависимости от того, какой файл вы редактировали

EXCEL_TO_LOAD = EXCEL_FILLED if EXCEL_FILLED.exists() else EXCEL_INPUT

# 🧪 ТЕСТОВАЯ БД: Для тестирования используем отдельную БД, чтобы не повредить основную
USE_TEST_DB = True  # Установите False для загрузки в основную БД
TEST_DB_PATH = COMPANY_ROOT / "data_mart_test.db"

# 🔄 АВТООБНОВЛЕНИЕ DICTIONARY_METRICS: автоматически обновить dictionary_metrics при обнаружении кастомных метрик
AUTO_UPDATE_DICTIONARY = False  # Установите True для автоматического обновления dictionary_metrics

if EXCEL_TO_LOAD.exists():
    load_cmd = [
        "python3",
        str(PROJECT_ROOT / "tools" / "load_excel_to_data_mart.py"),
        "--company", COMPANY,
        "--file", str(EXCEL_TO_LOAD),
        "--config", str(CONFIG_PATH) if CONFIG_PATH.exists() else str(PROJECT_ROOT / "configs" / "excel_loader.yaml")
    ]
    
    # Добавляем путь к тестовой БД, если включен режим тестирования
    if USE_TEST_DB:
        load_cmd.extend(["--test-db-path", str(TEST_DB_PATH)])
        print("🧪 РЕЖИМ ТЕСТИРОВАНИЯ: данные будут загружены в тестовую БД")
        print(f"   Тестовая БД: {TEST_DB_PATH}")
        print("   ⚠️ Основная БД не будет затронута")
    else:
        print("⚠️ РЕЖИМ ПРОДАКШЕН: данные будут загружены в основную БД")
        print("   Убедитесь, что вы хотите перезаписать данные!")
    
    # Добавляем флаг автоматического обновления dictionary_metrics
    if AUTO_UPDATE_DICTIONARY:
        load_cmd.append("--auto-update-dictionary")
        print("🔄 АВТООБНОВЛЕНИЕ DICTIONARY_METRICS: включено")
        print("   При обнаружении кастомных метрик они будут автоматически добавлены в dictionary_metrics")
    else:
        print("ℹ️ АВТООБНОВЛЕНИЕ DICTIONARY_METRICS: выключено")
        print("   Кастомные метрики будут обнаружены, но dictionary_metrics не будет обновлен автоматически")
        print("   Используйте флаг --auto-update-dictionary для автоматического обновления")
    
    print(f"\n🚀 Команда загрузки:")
    print(f"   {' '.join(load_cmd)}")
    print(f"📄 Файл: {EXCEL_TO_LOAD}")
    print("\n💡 НОВЫЕ ВОЗМОЖНОСТИ:")
    print("   - Автоматическое обнаружение кастомных метрик")
    print("   - Автоматическое предложение алиасов для новых метрик")
    print("   - Автоматическое обновление dictionary_metrics (опционально)")
    print("\n⚠️ Для реальной загрузки раскомментируйте код ниже:")
    
    # Раскомментируйте для реальной загрузки:
    # env = os.environ.copy()
    # env['PYTHONPATH'] = str(PROJECT_ROOT) + ':' + env.get('PYTHONPATH', '')
    # result = subprocess.run(load_cmd, cwd=PROJECT_ROOT, env=env, capture_output=True, text=True)
    # if result.returncode == 0:
    #     print("✅ Загрузка выполнена успешно!")
    #     if USE_TEST_DB:
    #         print(f"📊 Данные загружены в тестовую БД: {TEST_DB_PATH}")
    #     else:
    #         print("📊 Данные загружены в основную БД")
    #     print("\n🔍 Проверьте логи выше на наличие обнаруженных кастомных метрик:")
    #     print("   Ищите строки с '🔍 Обнаружена кастомная метрика'")
    # else:
    #     print("❌ Ошибка при загрузке:")
    #     print(result.stderr)
else:
    print(f"⚠️ Excel файл не найден: {EXCEL_TO_LOAD}")


## 3️⃣ Проверка структуры Excel файла

Проверяем структуру Excel файла и анализируем метрики.


In [ ]:
# Загружаем Excel файл для анализа
EXCEL_TO_ANALYZE = EXCEL_FILLED if EXCEL_FILLED.exists() else EXCEL_INPUT

if EXCEL_TO_ANALYZE.exists():
    book = pd.read_excel(EXCEL_TO_ANALYZE, sheet_name=None, dtype=object)
    print(f"📊 Найдено листов: {len(book)}")
    for name, df in book.items():
        print(f"   - {name:<30} rows={len(df):<5} cols={len(df.columns)}")
else:
    print(f"⚠️ Excel файл не найден: {EXCEL_TO_ANALYZE}")
    book = {}


## 4️⃣ Сравнение данных между основной и тестовой БД

После загрузки данных в тестовую БД, можно сравнить их с основной БД и запустить препроцесс для проверки.


In [ ]:
# Сравнение данных между основной и тестовой БД
# ВАЖНО: Убедитесь, что данные загружены в тестовую БД перед сравнением

if TEST_DB_PATH.exists():
    compare_cmd = [
        "python3",
        str(PROJECT_ROOT / "tools" / "compare_test_db.py"),
        "--company", COMPANY,
        "--test-db", str(TEST_DB_PATH),
        "--run-preprocess"  # Запустить препроцесс на тестовой БД
    ]
    
    print("🔍 Запуск сравнения данных и препроцесса...")
    print(f"Команда: {' '.join(compare_cmd)}")
    print(f"🧪 Тестовая БД: {TEST_DB_PATH}")
    print(f"📊 Основная БД: {PROJECT_ROOT / 'data_mart.db'}")
    print("\n⚠️ Для реального запуска раскомментируйте код ниже:")
    
    # Раскомментируйте для реального запуска:
    # env = os.environ.copy()
    # env['PYTHONPATH'] = str(PROJECT_ROOT) + ':' + env.get('PYTHONPATH', '')
    # result = subprocess.run(compare_cmd, cwd=PROJECT_ROOT, env=env, capture_output=True, text=True)
    # print(result.stdout)
    # if result.returncode != 0:
    #     print("❌ Ошибка при сравнении:")
    #     print(result.stderr)
else:
    print(f"⚠️ Тестовая БД не найдена: {TEST_DB_PATH}")
    print("   Сначала загрузите данные в тестовую БД (ячейка выше)")


In [ ]:
summary = []
for sheet_name, column in [
    ("history_IS", "metric"),
    ("history_BS", "metric"),
    ("history_CF", "metric"),
    ("segments_financial", "metric"),
    ("segments_operational", "metric"),
    ("operational_drivers", "driver_name"),
]:
    if sheet_name not in book:
        continue
    series = book[sheet_name][column].dropna().astype(str).str.strip()
    counts = Counter(series)
    summary.append({
        "sheet": sheet_name,
        "unique": len(counts),
        "top5": counts.most_common(5),
    })

pd.DataFrame(summary)


In [ ]:
if CONFIG_PATH.exists():
    loader_cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
else:
    loader_cfg = {}
loader_cfg.keys()


In [ ]:
def suggest_aliases(sheet: str, column: str = "metric") -> pd.DataFrame:
    if sheet not in book:
        return pd.DataFrame()
    series = book[sheet][column].dropna().astype(str).str.strip()
    counts = Counter(series)
    data = []
    for metric, count in counts.most_common():
        alias = metric.lower().replace(" ", "_")
        data.append({"sheet": sheet, "metric_raw": metric, "alias": alias, "count": count})
    return pd.DataFrame(data)

suggest_aliases("history_IS").head(10)


In [ ]:
aliases_df = pd.concat([
    suggest_aliases("history_IS"),
    suggest_aliases("history_BS"),
    suggest_aliases("history_CF"),
], ignore_index=True)

# оставляем только новые алиасы, которых нет в словаре
existing_aliases = set()
metrics_dict = loader_cfg.get("validation", {}).get("dictionaries", {}).get("metrics", {})
if metrics_dict:
    sheet_name = metrics_dict.get("sheet")
    aliases_column = metrics_dict.get("aliases_column", "accepted_aliases")
    key_column = metrics_dict.get("key_column", "canonical_metric")
    if sheet_name in book:
        df_dict = book[sheet_name]
        for _, row in df_dict.iterrows():
            canonical = str(row.get(key_column)).strip().lower()
            aliases_raw = str(row.get(aliases_column) or "")
            for alias in aliases_raw.split(";"):
                alias_norm = alias.strip().lower()
                if alias_norm:
                    existing_aliases.add(alias_norm)

new_aliases = aliases_df[~aliases_df["alias"].isin(existing_aliases)]
new_aliases.head(20)


In [ ]:
def build_yaml_alias_block(df: pd.DataFrame, statement: str) -> str:
    lines = [f"  {statement.upper()}:" ]
    for _, row in df.sort_values("metric_raw").iterrows():
        alias = row["alias"].strip()
        canonical = alias
        lines.append(f"    - source_metric: {alias}")
        lines.append(f"      target_metric: {canonical}")
        lines.append(f"      scale: auto")
        lines.append(f"      sign: auto")
    return "\n".join(lines)

build_yaml_alias_block(new_aliases[new_aliases["sheet"] == "history_IS"], "is")


## 4.5️⃣ Тестирование данных

Комплексное тестирование загруженных данных с использованием унифицированного интерфейса `test_suite.py`.


In [ ]:
# Комплексное тестирование данных
from tools.test_suite import TestSuite
from engine.notebook_helpers import display_test_results
from IPython.display import display, Markdown

# Настройки тестирования
TEST_YEARS = list(range(2010, 2025))  # Годы для тестирования
RUN_ALL_TESTS = True  # Запустить все тесты или только выбранные

print("🧪 Запуск комплексного тестирования данных...")
print(f"📊 Компания: {COMPANY}")
print(f"📅 Годы: {TEST_YEARS[0]}-{TEST_YEARS[-1]}")

# Создаем TestSuite
suite = TestSuite(company=COMPANY, root=PROJECT_ROOT, years=TEST_YEARS)

if RUN_ALL_TESTS:
    # Запускаем все тесты
    print("\n🚀 Запуск всех тестов...")
    result = suite.run_all_tests()
    
    # Отображаем результаты
    display_test_results({
        'total_tests': result.total_tests,
        'passed': result.passed,
        'failed': result.failed,
        'warnings': result.warnings,
        'skipped': result.skipped,
        'results': [
            {
                'name': r.name,
                'status': r.status,
                'message': r.message,
                'details': r.details
            }
            for r in result.results
        ]
    })
    
    # Сохраняем отчет
    report_path = COMPANY_ROOT / "outputs" / "test_report.json"
    report_path.parent.mkdir(parents=True, exist_ok=True)
    suite.generate_report(report_path)
    print(f"\n📄 Отчет сохранен: {report_path}")
else:
    # Запускаем отдельные тесты
    print("\n🔍 Запуск отдельных тестов...")
    
    # Тест полноты данных
    print("\n1️⃣ Тест полноты данных...")
    result_completeness = suite.test_data_completeness()
    print(f"   Статус: {result_completeness.status}")
    print(f"   Сообщение: {result_completeness.message}")
    
    # Тест мэппинга
    print("\n2️⃣ Тест мэппинга...")
    result_mapping = suite.test_mapping()
    print(f"   Статус: {result_mapping.status}")
    print(f"   Сообщение: {result_mapping.message}")
    
    # Тест подитогов
    print("\n3️⃣ Тест подитогов...")
    results_subtotals = suite.test_subtotals()
    for r in results_subtotals:
        print(f"   {r.name}: {r.status} - {r.message}")


## 4.6️⃣ Генерация рекомендаций по настройке YAML

Анализ результатов тестирования и автоматическая генерация рекомендаций по улучшению конфигурации `excel_loader.yaml`.


In [ ]:
# Генерация рекомендаций по настройке YAML
from tools.recommendation_generator import RecommendationGenerator
from engine.notebook_helpers import display_recommendations
from IPython.display import display, Markdown
import ipywidgets as widgets
from ipywidgets import HBox, VBox, Layout

print("🔍 Генерация рекомендаций по настройке YAML...")
print(f"📊 Компания: {COMPANY}")
print(f"📄 Конфиг: {CONFIG_PATH}")

# Создаем генератор рекомендаций
rec_gen = RecommendationGenerator(company=COMPANY, root=PROJECT_ROOT)

# Запускаем анализ
print("\n🚀 Запуск анализа...")
recommendations = rec_gen.analyze_all()

# Отображаем рекомендации
if recommendations:
    display(Markdown(f"### 📋 Найдено рекомендаций: {len(recommendations)}"))
    display_recommendations(recommendations)
    
    # Виджеты для применения рекомендаций
    display(Markdown("### ⚙️ Применение рекомендаций"))
    
    apply_high_priority = widgets.Button(
        description="✅ Применить рекомендации высокого приоритета",
        button_style='success',
        layout=Layout(width='400px')
    )
    
    apply_all = widgets.Button(
        description="✅ Применить все рекомендации",
        button_style='warning',
        layout=Layout(width='400px')
    )
    
    export_yaml = widgets.Button(
        description="📄 Экспортировать в YAML",
        button_style='info',
        layout=Layout(width='400px')
    )
    
    export_markdown = widgets.Button(
        description="📄 Экспортировать в Markdown",
        button_style='info',
        layout=Layout(width='400px')
    )
    
    output = widgets.Output()
    
    def on_apply_high(b):
        with output:
            output.clear_output()
            print("🔄 Применение рекомендаций высокого приоритета...")
            result = rec_gen.apply_recommendations(auto_apply=False, backup=True)
            print(f"✅ Применено: {result['applied']}")
            print(f"⏭️  Пропущено: {result['skipped']}")
            if result['errors']:
                print(f"❌ Ошибки: {len(result['errors'])}")
            if result.get('backup_path'):
                print(f"💾 Бэкап создан: {result['backup_path']}")
            display(Markdown("### ✅ Рекомендации применены!"))
    
    def on_apply_all(b):
        with output:
            output.clear_output()
            print("🔄 Применение всех рекомендаций...")
            result = rec_gen.apply_recommendations(auto_apply=True, backup=True)
            print(f"✅ Применено: {result['applied']}")
            print(f"⏭️  Пропущено: {result['skipped']}")
            if result['errors']:
                print(f"❌ Ошибки: {len(result['errors'])}")
            if result.get('backup_path'):
                print(f"💾 Бэкап создан: {result['backup_path']}")
            display(Markdown("### ✅ Все рекомендации применены!"))
    
    def on_export_yaml(b):
        with output:
            output.clear_output()
            export_path = COMPANY_ROOT / "outputs" / "recommendations.yaml"
            export_path.parent.mkdir(parents=True, exist_ok=True)
            rec_gen.export_recommendations(format="yaml", output_path=export_path)
            print(f"📄 Рекомендации экспортированы: {export_path}")
    
    def on_export_markdown(b):
        with output:
            output.clear_output()
            export_path = COMPANY_ROOT / "outputs" / "recommendations.md"
            export_path.parent.mkdir(parents=True, exist_ok=True)
            rec_gen.export_recommendations(format="markdown", output_path=export_path)
            print(f"📄 Рекомендации экспортированы: {export_path}")
    
    apply_high_priority.on_click(on_apply_high)
    apply_all.on_click(on_apply_all)
    export_yaml.on_click(on_export_yaml)
    export_markdown.on_click(on_export_markdown)
    
    display(VBox([
        HBox([apply_high_priority, apply_all]),
        HBox([export_yaml, export_markdown]),
        output
    ]))
else:
    display(Markdown("### ✅ Рекомендации отсутствуют"))
    print("Все настройки выглядят корректно!")


## 5️⃣ Итог и рекомендации

**После загрузки данных:**
1. Проверьте логи на наличие обнаруженных кастомных метрик (ищите `🔍 Обнаружена кастомная метрика`)
2. Если кастомные метрики обнаружены:
   - Проверьте предложенные алиасы (если включено автообновление, они уже добавлены в dictionary_metrics)
   - При необходимости скорректируйте алиасы вручную в Excel файле
   - Обновите `excel_loader.yaml` для добавления новых метрик в `required_metrics` (если нужно)
3. Скопируйте сгенерированный YAML-блок (если есть) в `configs/excel_loader.yaml`
4. Запустите `python tools/load_excel_to_data_mart.py --dry-run` для проверки
5. После успешной загрузки обновите документацию и синхронизируйте release

**Цветовая кодировка в шаблоне:**
- 🟡 Желтая подсветка = обязательные метрики (required_for_standard или required_for_custom)
- 🔵 Голубая подсветка = расчетные метрики (с combine_from правилами)
- ⚪ Белая = опциональные метрики

**Автоматическое обнаружение кастомных метрик:**
- Система автоматически обнаруживает метрики, которых нет в dictionary_metrics
- Для каждой новой метрики предлагаются похожие метрики из существующих (алиасы)
- При включении `--auto-update-dictionary` новые метрики автоматически добавляются в dictionary_metrics

